# Lab 9 - Data Cleaning
Automotive Industry Tracker - IT 2012 Unstructured Data

In [1]:
import sys
import os
sys.path.append(os.path.abspath("../src"))

Missing Value Analysis

In [2]:
from analytics.data_loader import load_from_csv
from cleaning.missing_handler import report_missing, save_missing_report

df = load_from_csv()
report = report_missing(df)
save_missing_report(df)

2026-05-04 15:55:43,499 - INFO - Loaded CSV: /Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/analytics/raw_export.csv, shape=(591, 56)
2026-05-04 15:55:43,501 - INFO - Missing value report: 55 columns with missing values
2026-05-04 15:55:43,504 - INFO - Missing value report: 55 columns with missing values
2026-05-04 15:55:43,507 - INFO - Missing report saved: /Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/cleaned/missing_report.csv


                           missing_count  missing_pct
data.text                            578        97.80
language_probability                 575        97.29
data.raw_text                        572        96.79
stored_at                            567        95.94
segment_count                        567        95.94
segments                             567        95.94
model                                567        95.94
duration                             567        95.94
language                             567        95.94
source_file                          567        95.94
data.processed_text                  550        93.06
data.paragraphs                      542        91.71
data.tables                          542        91.71
data.sheets                          542        91.71
data.pages                           532        90.02
data.exif.date_taken                 531        89.85
data.height                          531        89.85
data.file_size_kb           

'/Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/cleaned/missing_report.csv'

String Cleaning

In [3]:
from cleaning.string_cleaner import run_string_cleaning

df_clean = df.copy()
df_clean = run_string_cleaning(df_clean)
print(df_clean[["data.make", "source", "data.model"]].head())

2026-05-04 15:55:43,514 - INFO - Starting string cleaning
2026-05-04 15:55:43,515 - INFO - Cleaned title column: 'data.make'
2026-05-04 15:55:43,516 - INFO - Normalized source column: 'source'
2026-05-04 15:55:43,517 - INFO - Normalized model column: 'data.model'
2026-05-04 15:55:43,520 - INFO - Cleaned text column: 'data.description'
2026-05-04 15:55:43,530 - INFO - Extracted year from 'fetched_at' into 'release_year'
2026-05-04 15:55:43,535 - INFO - String cleaning complete


    data.make     source data.model
0        Audi       test         A4
1       Skoda  nhtsa_api    OCTAVIA
2  Volkswagen  nhtsa_api       GOLF
3        Audi  nhtsa_api         A4
4  Volkswagen  nhtsa_api     PASSAT


In [4]:
from analytics.regex_ops import detect_invalid_dates, detect_invalid_language_codes, extract_numbers_from_text, flag_short_overviews

detect_invalid_dates(df)
detect_invalid_language_codes(df)
extract_numbers_from_text(df)
flag_short_overviews(df)

2026-05-04 15:55:43,554 - INFO - Invalid dates in 'fetched_at': 0
2026-05-04 15:55:43,555 - INFO - Invalid language codes in 'language': 0
2026-05-04 15:55:43,556 - INFO - Extracted numbers from text in 80 rows
2026-05-04 15:55:43,557 - INFO - Short overviews (< 30 chars): 0


Invalid dates in 'fetched_at': 0
Invalid language codes in 'language': 0
Rows with numbers in text: 80
Short overviews found: 0


,source,fetched_at,version,_collection,data.make,data.model,data.year,data.recalls,data.file_name,data.document_type,...,data.exif.extracted_at,data.processed_at,source_file,language,language_probability,duration,model,segments,segment_count,stored_at


Deduplication

In [5]:
from cleaning.deduplicator import count_duplicates, drop_exact_duplicates, drop_duplicate_ids

print("Rows before:", len(df_clean))
count_duplicates(df_clean)
df_clean = drop_exact_duplicates(df_clean)
count_duplicates(df_clean, col="source")
df_clean = drop_duplicate_ids(df_clean, id_col="source")
print("Rows after:", len(df_clean))

2026-05-04 15:55:43,569 - INFO - Exact duplicate rows: 0
2026-05-04 15:55:43,572 - INFO - Dropped 0 exact duplicate rows
2026-05-04 15:55:43,573 - INFO - Duplicate values in 'source': 579
2026-05-04 15:55:43,573 - INFO - Dropped 579 rows with duplicate 'source'


Rows before: 591
Exact duplicate rows: 0
Rows before: 591, after dropping exact duplicates: 591
Duplicate values in 'source': 579
Rows after dropping duplicate 'source': 12
Rows after: 12


Type Conversion

In [6]:
from cleaning.type_converter import run_type_conversion

df_before = df_clean.copy()
df_clean = run_type_conversion(df_clean)
print(df_clean.dtypes)

2026-05-04 15:55:43,577 - INFO - Starting type conversion
2026-05-04 15:55:43,579 - INFO - Converted 'fetched_at' to datetime
2026-05-04 15:55:43,580 - INFO - Converted 'data.scraped_at' to datetime
2026-05-04 15:55:43,581 - INFO - Converted 'data.processed_at' to datetime
2026-05-04 15:55:43,582 - INFO - Converted 'stored_at' to datetime
2026-05-04 15:55:43,583 - INFO - Converted 'version' to float32
2026-05-04 15:55:43,583 - INFO - Converted 'data.file_size_kb' to float32
2026-05-04 15:55:43,584 - INFO - Converted 'data.width' to float32
2026-05-04 15:55:43,585 - INFO - Converted 'data.height' to float32
2026-05-04 15:55:43,585 - INFO - Converted 'source' to category
2026-05-04 15:55:43,586 - INFO - Converted '_collection' to category
2026-05-04 15:55:43,587 - INFO - Converted 'data.make' to category
2026-05-04 15:55:43,587 - INFO - Converted 'data.model' to category
2026-05-04 15:55:43,588 - INFO - Converted 'data.type' to category
2026-05-04 15:55:43,591 - INFO - Memory report - be

Memory before : 0.04 MB
Memory after  : 0.04 MB
Reduction     : 0.00 MB
source                             category
fetched_at                   datetime64[ns]
version                             float32
_collection                        category
data.make                          category
data.model                         category
data.year                           float64
data.recalls                         object
data.file_name                       object
data.document_type                   object
data.source                          object
data.extraction_timestamp            object
data.extraction_library              object
data.pages                           object
data.paragraphs                      object
data.tables                          object
data.sheets                          object
data.scraped_at              datetime64[ns]
data.type                          category
data.raw_text                        object
data.processed_text                  object
data

Validation

In [7]:
from cleaning.validator import run_validation

run_validation(df_clean)

2026-05-04 15:55:43,596 - INFO - Starting validation
2026-05-04 15:55:43,596 - INFO - Validation passed: 'source' has no null values
2026-05-04 15:55:43,597 - INFO - Validation passed: '_collection' has no null values
2026-05-04 15:55:43,597 - INFO - Validation passed: all years in 'release_year' are within [1900, 2030]
2026-05-04 15:55:43,598 - INFO - Validation passed: 'fetched_at' dtype is datetime64[ns]
2026-05-04 15:55:43,598 - INFO - Validation passed: 'version' dtype is float32
2026-05-04 15:55:43,599 - INFO - All validations passed


Validation passed: no null values in critical columns
Validation passed: year range [1900, 2030]
Validation passed: all column types correct
All validations passed


Full Cleaning Pipeline

In [8]:
from cleaning.clean_pipeline import run_cleaning_pipeline

df_raw = load_from_csv()
df_final = run_cleaning_pipeline(df_raw)
print(df_final.shape)

2026-05-04 15:55:43,611 - INFO - Loaded CSV: /Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/analytics/raw_export.csv, shape=(591, 56)
2026-05-04 15:55:43,612 - INFO - Starting cleaning pipeline
2026-05-04 15:55:43,613 - INFO - Step 1: Dropping high missing columns
2026-05-04 15:55:43,617 - INFO - Dropped 15 columns with >90.0% missing: ['data.pages', 'data.paragraphs', 'data.tables', 'data.sheets', 'data.raw_text', 'data.processed_text', 'data.text', 'source_file', 'language', 'language_probability', 'duration', 'model', 'segments', 'segment_count', 'stored_at']
2026-05-04 15:55:43,617 - INFO - Step 2: Dropping rows with missing critical columns
2026-05-04 15:55:43,619 - INFO - Dropped 24 rows with missing critical columns: ['source', '_collection']
2026-05-04 15:55:43,619 - INFO - Step 3: Filling text fields
2026-05-04 15:55:43,620 - INFO - Filled missing text in 'data.make' with 'unknown'
2026-05-04 15:55:43,621 - INFO - Fill

Dropped columns: ['data.pages', 'data.paragraphs', 'data.tables', 'data.sheets', 'data.raw_text', 'data.processed_text', 'data.text', 'source_file', 'language', 'language_probability', 'duration', 'model', 'segments', 'segment_count', 'stored_at']
Dropped 24 rows missing critical columns
Rows before deduplication: 567
Exact duplicate rows: 0
Rows before: 567, after dropping exact duplicates: 567
Duplicate values in 'source': 556
Rows after dropping duplicate 'source': 11
Rows after dropping duplicate title+date: 11
Memory before : 0.02 MB
Memory after  : 0.02 MB
Reduction     : 0.00 MB
Validation passed: no null values in critical columns
Validation passed: year range [1900, 2030]
Validation passed: all column types correct
All validations passed
Cleaned dataset saved: /Users/elmedin.karisiksystemverification.com/Documents/private/Automotive-Industry-Tracker/data/processed/cleaned/cleaned_data.csv
Final shape: (11, 42)
(11, 42)
